In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
import numpy as np 


data_train = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')
data_test = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/test.csv')

In [2]:
data_train.head(10)

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True
5,0005_01,Earth,False,F/0/P,PSO J318.5-22,44.0,False,0.0,483.0,0.0,291.0,0.0,Sandie Hinetthews,True
6,0006_01,Earth,False,F/2/S,TRAPPIST-1e,26.0,False,42.0,1539.0,3.0,0.0,0.0,Billex Jacostaffey,True
7,0006_02,Earth,True,G/0/S,TRAPPIST-1e,28.0,False,0.0,0.0,0.0,0.0,NaN,Candra Jacostaffey,True
8,0007_01,Earth,False,F/3/S,TRAPPIST-1e,35.0,False,0.0,785.0,17.0,216.0,0.0,Andona Beston,True
9,0008_01,Europa,True,B/1/P,55 Cancri e,14.0,False,0.0,0.0,0.0,0.0,0.0,Erraiam Flatic,True


In [3]:
data_train.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [4]:
def create_features(df, is_train=True):
    df = df.copy()
    df[['CabinDeck', 'CabinNum', 'CabinSide']] = df['Cabin'].str.split('/', expand=True)
    return df

In [5]:
X_train = create_features(data_train, is_train=True)
X_test = create_features(data_test, is_train=False)

In [6]:
num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in num_cols:
    median_val = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_val)
    X_test[col] = X_test[col].fillna(median_val)

In [7]:
cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'CabinDeck', 'CabinSide']
for col in cat_cols:
    mode_val = X_train[col].mode()[0]
    X_train[col] = X_train[col].fillna(mode_val)
    X_test[col] = X_test[col].fillna(mode_val)

/tmp/ipykernel_16/1708133855.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train[col] = X_train[col].fillna(mode_val)
/tmp/ipykernel_16/1708133855.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_test[col] = X_test[col].fillna(mode_val)


In [8]:
X_train.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,CabinDeck,CabinNum,CabinSide
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False,B,0,P
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True,F,0,S
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False,A,0,S
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False,A,0,S
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True,F,1,S


In [9]:
X_train = X_train.drop(['PassengerId', 'Transported', 'Name', 'Cabin', 'CabinNum'], axis = 1)
y_train = data_train.Transported


In [10]:
X_test  = X_test.drop(['PassengerId', 'Name', 'Cabin', 'CabinNum'], axis = 1)

In [11]:
X_train = pd.get_dummies(X_train, dtype=int)
X_test = pd.get_dummies(X_test, dtype=int)
X_train, X_test = X_train.align(X_test, join='inner', axis=1)

In [12]:
clf = RandomForestClassifier(random_state=42, n_jobs=-1)
parameters = {
    'n_estimators': [50, 100, 150],
    'max_depth': [5, 7],
    'min_samples_split': [2, 5]
}
grid_search_CV_clf = GridSearchCV(clf, parameters, scoring='accuracy', cv = 5, n_jobs=-1)
grid_search_CV_clf.fit(X_train,y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(n_jobs=-1, random_state=42),
             n_jobs=-1,
             param_grid={'max_depth': [5, 7], 'min_samples_split': [2, 5],
                         'n_estimators': [50, 100, 150]},
             scoring='accuracy')

In [13]:
print(f" Best CV-score: {grid_search_CV_clf.best_score_:.4f}")
print(f" Params: {grid_search_CV_clf.best_params_}")

 Best CV-score: 0.7920
 Params: {'max_depth': 7, 'min_samples_split': 5, 'n_estimators': 100}


In [14]:
best_clf = grid_search_CV_clf.best_estimator_
feature_importances = best_clf.feature_importances_
feature_importances_df = pd.DataFrame({'features' :list(X_train), 'feature_importances' : feature_importances})
feature_importances_df.sort_values('feature_importances', ascending = False)

,features,feature_importances
0,CryoSleep,0.186918
2,RoomService,0.164706
5,Spa,0.129620
6,VRDeck,0.126603
3,FoodCourt,0.103419
4,ShoppingMall,0.077430
7,HomePlanet_Earth,0.050040
1,Age,0.029615
8,HomePlanet_Europa,0.026514
21,CabinDeck_G,0.014548


In [15]:
predictions = grid_search_CV_clf.predict(X_test)

In [16]:
submission = pd.DataFrame({
    'PassengerId': data_test['PassengerId'],
    'Transported': predictions.astype(bool)   
})
submission.to_csv('submission.csv', index=False)